<a href="https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I chose this lane because it matches a real, concrete decision: *which page should a content reviewer look at first?* Reviewers have limited time, so the order of the list matters more than a yes/no label. The starter pipeline already gives me a transparent rule baseline to beat, and the starter data shows that impressions are very concentrated in a small share of pages (numbers in section 3), which is exactly when a good ordering is worth building.

My backup option is Lane 4 (CTR / engagement opportunity), because most visible pages have a very low CTR. I can confirm or change my lane until the end of Week 4.

In [5]:
import os, sys, subprocess
import pandas as pd

# Find the repo root so the CSV loads in Colab and locally.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yelsebety/Yassin-Flyrank.git"
REPO_DIR = "Yassin-Flyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Starter data: {len(df):,} rows x {df.shape[1]} columns, {df['client_id'].nunique()} clients")
print("One row = one content item (content_id), trailing-90-day metrics.")

Starter data: 30,000 rows x 44 columns, 32 clients
One row = one content item (content_id), trailing-90-day metrics.


## 2. The question: decision, action, cost of a wrong call

**Search question:** Among pages that already get search impressions, which ones should a content reviewer look at *first* because they show signs of losing or under-capturing visibility?

**Unit of analysis:** one content item (a page). In the starter data, one row per `content_id`.

**Output:** a ranked review queue — a score per page plus short reason codes (for example *declining with demand*, *stale visible page*) so a human can see why a page ranks high.

**Decision and action:** a content reviewer or SEO editor decides which pages to spend limited time on. They review the top K pages (for example the top 20 or 50) and choose an action for each: refresh, expand, protect, prune, or monitor.

**Cost of a wrong call:**
- *False alarm* (a page in the top K that was fine): wasted editor hours, and a reviewer who stops trusting the list.
- *Missed page* (a real problem ranked low): a page with real demand stays unreviewed for longer.
- With limited capacity, false alarms at the top of the list hurt most right away, so **precision@K** is the metric I care about first. A recommendation is a reason to *look*, not a promise that a refresh will work.

**Why data or ML can help at all:** the signals (impressions, position, CTR, age, freshness, sessions, content type) interact, and a single hand-written rule only captures part of that. In the starter pipeline's committed report (`outputs/model_report.md`), the rule baseline got a precision@50 of 0.240 and the best model (random forest, client-holdout split) got 0.740. That *suggests* learning can beat a fixed rule on this slice, but the label there is a proxy (`trend_direction == "down"`), so I would have to earn the result again with a future-window outcome. If a plain rule turns out to be just as good, that is a valid finding too.

**One-paragraph frame:** For a content reviewer deciding which pages to review first, I will build a ranked review queue with reason codes from observable search and engagement signals, scoring pages by decline risk and measuring it with precision@K against a simple rule baseline. A wrong call costs wasted review time or a missed declining page. A plain rule isn't enough because many signals interact. I will claim only observed, directional, decision-support results.

In [6]:
# How much of the review work can a small list cover? (reviewer capacity check)
s = df["impressions_90d"].sort_values(ascending=False)
total = s.sum()

top50_share = s.head(50).sum() / total
pages_for_half = int((s.cumsum() / total < 0.5).sum() + 1)

print(f"Top 50 pages hold {top50_share:.1%} of all 90-day impressions.")
print(f"{pages_for_half:,} pages ({pages_for_half/len(s):.1%} of {len(s):,}) hold half of all impressions.")
print(f"50 pages are only {50/len(s):.2%} of all pages, so a short list is a small share of the visibility. That is why the ordering of the list matters.")

Top 50 pages hold 8.3% of all 90-day impressions.
1,244 pages (4.1% of 30,000) hold half of all impressions.
50 pages are only 0.17% of all pages, so a short list is a small share of the visibility. That is why the ordering of the list matters.


## 3. Quick look at the data (2-3 real numbers)

These numbers come from the 30,000-row starter dataset (one 90-day window, 32 clients). They are about this sample only.

1. **Impressions are concentrated.** A small share of pages holds most of the visibility, so ordering the review list well matters.
2. **A lot of pages are flagged "down".** More than half of pages have `trend_direction == "down"`, so a useful ranking has to do better than just "everything is declining" (the base rate).
3. **Many visible pages have low CTR.** That makes a single fixed CTR rule too blunt on its own, which is where a position-aware score could help (and is my Lane 4 backup).

In [7]:
# Number 1: concentration of impressions
s = df["impressions_90d"].sort_values(ascending=False)
top10_share = s.head(int(len(s) * 0.10)).sum() / s.sum()
print(f"1) Top 10% of pages hold {top10_share:.1%} of all 90-day impressions.")

# Number 2: base rate of the 'down' label (this is a PROXY label, not a future outcome)
down_rate = (df["trend_direction"] == "down").mean()
big = df[df["impressions_90d"] >= 500]
down_rate_big = (big["trend_direction"] == "down").mean()
print(f"2) {down_rate:.1%} of all pages are 'down'; {down_rate_big:.1%} among pages with 500+ impressions.")

# Number 3: low CTR among visible pages (ctr is a x100 percentage: 0.5 means 0.5%; avg_position == 0 means no data)
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)]
low_ctr = (visible["ctr"] < 0.5).mean()
print(f"3) {len(visible):,} visible pages (500+ impressions, position 1-20); {low_ctr:.1%} have CTR below 0.5%.")

1) Top 10% of pages hold 70.2% of all 90-day impressions.
2) 54.2% of all pages are 'down'; 59.6% among pages with 500+ impressions.
3) 12,023 visible pages (500+ impressions, position 1-20); 81.2% have CTR below 0.5%.


## 4. Careful words: what I can and can't claim

**What my work will be able to say**
- *Observed:* what the data shows in this sample (one 90-day window, 32 clients), such as which signals go together with decline.
- *Directional:* that one method ranks pages better than another on my validation setup, and by how much.
- *Decision-support:* that a ranked list can help a reviewer spend limited time on the most promising pages first, with reasons they can inspect.

**What my work will never say**
- That a refresh *caused* a recovery. This data is observational, with no experiment, so I cannot show cause and effect.
- That I found or "predicted" Google's algorithm or any ranking factor.
- That results from the 30,000-row starter slice carry over to the full warehouse. They would have to be earned again there.
- That "declining" is a guaranteed future outcome. In the starter data, the label comes from the current window, so it is a proxy.

**Rules I will follow:** `trend_direction` and `trend_pct` are never features (the label comes from them). IDs are only for grouping and splitting. Position 0 means no data, not rank zero. No client names, URLs, or private queries appear anywhere.

In [8]:
# Guard rails I will keep for the rest of the project
NEVER_FEATURES = ["trend_direction", "trend_pct", "content_id", "client_id"]
print("Never used as features:", NEVER_FEATURES)

no_pos = (df["avg_position"] == 0).sum()
print(f"avg_position == 0 means 'no data': {no_pos:,} rows ({no_pos/len(df):.1%}) are excluded from position and CTR numbers above.")

base = (df["trend_direction"] == "down").mean()
print(f"Base rate for the proxy label is {base:.1%}, so precision@K has to beat that to mean anything.")

Never used as features: ['trend_direction', 'trend_pct', 'content_id', 'client_id']
avg_position == 0 means 'no data': 1,205 rows (4.0%) are excluded from position and CTR numbers above.
Base rate for the proxy label is 54.2%, so precision@K has to beat that to mean anything.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.